# RAG Pipeline — Tegas-Gagas
RAG atas 4 dokumen UU: Parent-Child chunking, metadata enrichment, Ensemble Retriever (BM25 + FAISS), HyDE, Reranker + fallback DuckDuckGo Search.

> Gunakan model hasil `Fine-tuning_submission` (atau `GRPO_submission` bila dijalankan) sebagai model generation.


## 1. Setup & Install

In [1]:
!pip install -q unsloth
!pip install -q langchain langchain-community sentence-transformers faiss-cpu pypdf rank_bm25
!pip install -q gradio huggingface_hub duckduckgo-search


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.6/72.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [18]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [2]:
import os, re, uuid, torch
from unsloth import FastLanguageModel

FT_REPO_NAME = "noviardhana/llama3-legal-id-grpo"     # atau ganti ke GRPO_REPO_NAME jika sudah GRPO
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = FT_REPO_NAME,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load noviardhana/llama3-legal-id-grpo as a legacy tokenizer.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Ll

## 2. Load Dokumen PDF (Basic)
Unduh 4 file PDF UU dari folder Google Drive yang diberikan, letakkan di `./legal_docs/`.

In [3]:
LEGAL_DOCS_DIR = "legal_docs"
os.makedirs(LEGAL_DOCS_DIR, exist_ok=True)

!pip install -q gdown
!gdown --folder "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql" -O legal_docs

pdf_files = [f for f in os.listdir(LEGAL_DOCS_DIR) if f.lower().endswith(".pdf")]
print("Dokumen ditemukan:", pdf_files)
assert len(pdf_files) == 4, "Wajib menggunakan seluruh 4 dokumen UU"

Retrieving folder contents
Processing file 1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU PP Nomor 5 Tahun 2021.pdf
Processing file 1trvqHE72Anu8MhsNvvYJHieWqbao2BCy PP Nomor 35 Tahun 2021.pdf
Processing file 1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT PP Nomor 51 Tahun 2023.pdf
Processing file 1jf5f9ZHF2tzcK7Mm7eHCkDEtVntQ-y1s UU Nomor 6 Tahun 2023.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU
To: /content/legal_docs/PP Nomor 5 Tahun 2021.pdf
100% 17.1M/17.1M [00:00<00:00, 37.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1trvqHE72Anu8MhsNvvYJHieWqbao2BCy
To: /content/legal_docs/PP Nomor 35 Tahun 2021.pdf
100% 2.52M/2.52M [00:00<00:00, 254MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT
To: /content/legal_docs/PP Nomor 51 Tahun 2023.pdf
100% 2.77M/2.77M [00:00<00:00, 191MB/s]
Downloading...
From: https://dr

## 3. Text Splitting + Metadata Enrichment + Parent-Child Chunking (Basic + Skilled)

- `parent_splitter`: chunk besar (halaman/potongan utuh) — jadi konteks LLM
- `child_splitter`: chunk kecil — jadi target pencarian vektor
- Metadata enrichment: nomor & tahun UU diekstrak dari teks dokumen

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=300,  chunk_overlap=50)

def extract_uu_metadata(text):
    m = re.search(r"UNDANG[- ]?UNDANG\s+(?:REPUBLIK INDONESIA\s+)?NOMOR\s+(\d+)\s+TAHUN\s+(\d{4})", text.upper())
    if m:
        return {"uu_nomor": m.group(1), "uu_tahun": m.group(2)}
    return {"uu_nomor": "unknown", "uu_tahun": "unknown"}

parent_docs = {}
child_docs = []

for fname in pdf_files:
    loader = PyPDFLoader(os.path.join(LEGAL_DOCS_DIR, fname))
    pages = loader.load()
    full_text = "\n".join(p.page_content for p in pages)
    uu_meta = extract_uu_metadata(full_text[:3000])

    parents = parent_splitter.create_documents([full_text])
    for p_idx, parent in enumerate(parents):
        parent_id = str(uuid.uuid4())
        parent.metadata.update({"source": fname, "parent_idx": p_idx, **uu_meta})
        parent_docs[parent_id] = parent

        children = child_splitter.create_documents([parent.page_content])
        for c_idx, child in enumerate(children):
            child.metadata.update({
                "source": fname, "parent_id": parent_id,
                "parent_idx": p_idx, "child_idx": c_idx, **uu_meta,
            })
            child_docs.append(child)

print(f"Total parent chunks: {len(parent_docs)} | Total child chunks: {len(child_docs)}")
print(child_docs[0].page_content[:200])
print(child_docs[0].metadata)


Total parent chunks: 1092 | Total child chunks: 8923
SALINAN
PRESIDEN
NEPUBUK INDONESIA
UNDANG-UNDANG REPUBLIK INDONESIA
NOMOR 6 TAHUN 2023
TENTANG
PENETAPAN PERATURAN PEMERINTAH PENGGANTI UNDANG-UNDANG
NOMOR 2 TAHUN 2022 TENTANG CIPTA KERJA
MENJADI UND
{'source': 'UU Nomor 6 Tahun 2023.pdf', 'parent_id': 'e3f52542-b553-4bb2-956b-7a6d9127b983', 'parent_idx': 0, 'child_idx': 0, 'uu_nomor': '6', 'uu_tahun': '2023'}


## 4. Embedding + Vector DB (FAISS) + Ensemble Retriever (Basic + Skilled)

Embedding open-source multilingual, disimpan di FAISS lokal. Ensemble menggabungkan BM25 (keyword) + FAISS (semantik) dengan bobot eksplisit, retrieve ≥5 dokumen.

In [9]:
# Ganti import: pakai package baru langchain-huggingface, bukan langchain_community
!pip install -q -U langchain-huggingface

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever

try:
    from langchain.retrievers import EnsembleRetriever
except ModuleNotFoundError:
    from langchain_classic.retrievers import EnsembleRetriever

from sentence_transformers import SentenceTransformer
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
vectorstore = FAISS.from_documents(child_docs, embeddings)
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

bm25_retriever = BM25Retriever.from_documents(child_docs)
bm25_retriever.k = 5

ensemble_retriever = EnsembleRetriever(
    retrievers=[semantic_retriever, bm25_retriever],
    weights=[0.6, 0.4],
)

def retrieve_parent_chunks(query, metadata_filter=None):
    child_hits = ensemble_retriever.invoke(query)
    seen, parents = set(), []
    for c in child_hits:
        if metadata_filter and not all(c.metadata.get(k) == v for k, v in metadata_filter.items()):
            continue
        pid = c.metadata["parent_id"]
        if pid not in seen:
            seen.add(pid)
            parents.append(parent_docs[pid])
    return parents

test_parents = retrieve_parent_chunks("Berapa jam maksimal lembur dalam sehari?")
print(len(test_parents), "parent chunks ditemukan")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


6 parent chunks ditemukan


## 5. HyDE — Hypothetical Document Embeddings (Advanced)

In [10]:
model.generation_config.max_length = None

def generate_hyde_answers(question, n=2, max_new_tokens=150):
    """Buat n jawaban halusinasi dari LLM (belum melihat dokumen) untuk memperkaya query retrieval."""
    hypotheticals = []
    for _ in range(n):
        p = f"Jawablah pertanyaan hukum berikut secara singkat dan meyakinkan (boleh spekulatif): {question}"
        messages = [{"role": "user", "content": p}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.9)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        hypotheticals.append(text.strip())
    return hypotheticals

def hyde_retrieve(question, n_hyde=2, metadata_filter=None):
    hypotheticals = generate_hyde_answers(question, n=n_hyde)
    combined_query = question + "\n" + "\n".join(hypotheticals)
    return retrieve_parent_chunks(combined_query, metadata_filter=metadata_filter), hypotheticals

parents, hyde_texts = hyde_retrieve("Saya staf admin lembur 3 jam, apakah berhak dapat uang lembur?")
print("Jumlah jawaban halusinasi:", len(hyde_texts))
for h in hyde_texts:
    print("-", h[:150])


Jumlah jawaban halusinasi: 2
- Ya, ada beberapa pendapat yang mengatakan bahwa staf lembur berhak mendapatkan uang lembur karena mereka menghabiskan waktu untuk bisnis dari waktu pu
- Pada dasarnya, sebagai staf yang bekerja lembur, karyawan memiliki hak untuk mendapatkan uang lembur atas waktu tambahan yang mereka lakukan selain da


## 6. Reranker (Cross-Encoder) + Fallback DuckDuckGo Search (Advanced)

In [19]:
!pip install -q -U ddgs


In [20]:
import logging
from sentence_transformers import CrossEncoder
from ddgs import DDGS   # ganti dari: from duckduckgo_search import DDGS

logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
RERANK_TOP_K = 3
RELEVANCE_THRESHOLD = 0.0   # kalibrasi setelah lihat distribusi skor pada data Anda

def rerank(question, candidate_docs, top_k=RERANK_TOP_K):
    pairs = [(question, d.page_content) for d in candidate_docs]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(candidate_docs, scores), key=lambda x: x[1], reverse=True)
    top = ranked[:top_k]
    top1_score = float(top[0][1]) if top else float("-inf")
    return top, top1_score

def web_fallback(question, max_results=3):
    try:
        results = DDGS().text(question, max_results=max_results)
        return [f"[web: {r.get('href','')}] {r.get('title','')} - {r.get('body','')}" for r in results]
    except Exception:
        return []  # senyapkan error rate-limit DDG, biar tidak crash pipeline

def get_context(question, n_hyde=2, metadata_filter=None, verbose=False):
    parents, _ = hyde_retrieve(question, n_hyde=n_hyde, metadata_filter=metadata_filter)
    if not parents:
        return web_fallback(question), []

    top_ranked, top1_score = rerank(question, parents)
    if verbose:
        print(f"Reranker Top-1 score: {top1_score:.4f}")

    if top1_score < RELEVANCE_THRESHOLD:
        if verbose:
            print("Skor di bawah threshold -> fallback ke DuckDuckGo Search")
        return web_fallback(question), []

    context_texts, citations = [], []
    for doc, score in top_ranked:
        meta = doc.metadata
        citations.append(f"{meta.get('source')} (UU {meta.get('uu_nomor')}/{meta.get('uu_tahun')})")
        context_texts.append(f"[{meta.get('source')}] {doc.page_content}")
    return context_texts, citations

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 7. Prompt {context}/{question} + Generation dengan Sitasi (Basic + Skilled)

In [21]:
RAG_PROMPT_TEMPLATE = """Kamu adalah asisten hukum. Jawab pertanyaan HANYA berdasarkan konteks berikut.
Sertakan sitasi sumber di akhir jawaban.

Konteks:
{context}

Pertanyaan: {question}

Jawaban:"""

def rag_answer(question, max_new_tokens=400):
    context_texts, citations = get_context(question)
    context = "\n\n".join(context_texts)
    filled_prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)

    messages = [{"role": "user", "content": filled_prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    if citations:
        answer += "\n\nSumber: " + "; ".join(sorted(set(citations)))
    return answer

# Test Case Wajib
print(rag_answer("Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"))


Saya tidak dapat memberikan saran hukum. Namun, menurut Undang-Undang Nomor 13 Tahun 2003 tentang Pengelolaan Pajak, jika Anda bekerja lebih dari 40 jam dalam seminggu, Anda berhak mendapatkan uang lembur. Jika Anda bekerja lebih dari 40 jam dalam seminggu, Anda harus mendapatkan uang lembur untuk setiap jam yang berlebihan. Jadi, dalam kasus Anda, jika Anda bekerja 43 jam dalam seminggu, Anda berhak mendapatkan uang lembur untuk 3 jam tambahan.


## 8. Interface

In [22]:
import gradio as gr

demo = gr.Interface(
    fn=rag_answer,
    inputs=gr.Textbox(label="Pertanyaan", placeholder="Tulis pertanyaan hukum Anda..."),
    outputs=gr.Textbox(label="Jawaban"),
    title="RAG Legal Assistant",
)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3d6bf38d6f638845b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
